In [1]:
import subprocess
import sys

# Check which tools are available
for cmd in ["uv", "gh", "pip-audit", "pip"]:
    result = subprocess.run(["which", cmd], capture_output=True, text=True)
    print(f"{cmd}: {result.stdout.strip() or 'NOT FOUND'}")

# Check GitHub CLI run failures
result = subprocess.run(
    ["gh", "run", "list", "--limit", "5", "--status", "failure", "--repo", "systemslibrarian/meow-decoder"],
    capture_output=True, text=True, cwd="/workspaces/meow-decoder"
)
print("\nFailed runs:")
print(result.stdout or result.stderr)


uv: NOT FOUND
gh: /usr/bin/gh
pip-audit: NOT FOUND
pip: /workspaces/meow-decoder/.venv/bin/pip

Failed runs:
completed	failure	docs: remove previous release APK link from README	Fuzzing	main	push	22456719137	34m5s	2026-02-26T18:58:24Z
completed	failure	docs: remove previous release APK link from README	Security CI	main	push	22456719093	10m58s	2026-02-26T18:58:24Z
completed	failure	docs: remove previous release APK link from README	.github/workflows/rust-test-coverage.yml	main	push	22456718082	0s	2026-02-26T18:58:22Z
completed	failure	docs: add Android APK download links + iOS/store coming-soon notices …	Security CI	main	push	22456687745	10m46s	2026-02-26T18:57:33Z
completed	failure	docs: add Android APK download links + iOS/store coming-soon notices …	Fuzzing	main	push	22456683251	33m54s	2026-02-26T18:57:26Z



In [2]:
import subprocess

# Get logs for the failing workflows
# rust-test-coverage.yml failing at 0s (run ID 22456718082)
result = subprocess.run(
    ["gh", "run", "view", "22456718082", "--log-failed"],
    capture_output=True, text=True, cwd="/workspaces/meow-decoder"
)
print("=== rust-test-coverage.yml 0s failure ===")
print(result.stdout[:3000] if result.stdout else result.stderr[:3000])


=== rust-test-coverage.yml 0s failure ===
failed to get run log: log not found



In [3]:
import subprocess

# Get details on the 0s failure
result = subprocess.run(
    ["gh", "run", "view", "22456718082"],
    capture_output=True, text=True, cwd="/workspaces/meow-decoder"
)
print("=== rust-test-coverage.yml details ===")
print(result.stdout or result.stderr)

# Get logs for Security CI failure
result2 = subprocess.run(
    ["gh", "run", "view", "22456719093", "--log-failed"],
    capture_output=True, text=True, cwd="/workspaces/meow-decoder"
)
print("\n=== Security CI failure logs ===")
print(result2.stdout[:5000] if result2.stdout else result2.stderr[:2000])


=== rust-test-coverage.yml details ===

X main .github/workflows/rust-test-coverage.yml · 22456718082
Triggered via push about 48 minutes ago

X This run likely failed because of a workflow file issue.

For more information, see: https://github.com/systemslibrarian/meow-decoder/actions/runs/22456718082


=== Security CI failure logs ===
security	Security-focused tests (required)	﻿2026-02-26T19:00:14.9883602Z ##[group]Run pytest tests/ -v -m "security or adversarial" || {
security	Security-focused tests (required)	2026-02-26T19:00:14.9884222Z pytest tests/ -v -m "security or adversarial" || {
security	Security-focused tests (required)	2026-02-26T19:00:14.9884537Z   echo "Security tests failed!"
security	Security-focused tests (required)	2026-02-26T19:00:14.9884772Z   exit 1
security	Security-focused tests (required)	2026-02-26T19:00:14.9884931Z }
security	Security-focused tests (required)	2026-02-26T19:00:14.9933619Z shell: /usr/bin/bash -e {0}
security	Security-focused tests (required)

In [4]:
import subprocess

# Get more of the Security CI failure (get last part of security-focused tests)
result = subprocess.run(
    ["gh", "run", "view", "22456719093", "--log-failed"],
    capture_output=True, text=True, cwd="/workspaces/meow-decoder"
)
# Print the last part
log = result.stdout or result.stderr
lines = log.split('\n')
print(f"Total lines: {len(lines)}")
# Print the last 100 lines to see what failed
print('\n'.join(lines[-100:]))


Total lines: 4045
security	Security-focused tests (required)	2026-02-26T19:09:16.6177538Z meow_decoder/crypto_backend.py             149     26      6      1    83%   33-34, 186-191, 415, 427, 467, 473, 490, 501, 505, 509, 513, 517, 521, 525, 529, 533, 539, 543, 547, 563, 568, 594, 620, 657
security	Security-focused tests (required)	2026-02-26T19:09:16.6178192Z meow_decoder/decode_gif.py                 516    109    182     22    77%   99->102, 224-226, 233-234, 468-471, 521-523, 564, 601, 654, 658, 697-701, 721-722, 727, 735, 739, 819-846, 876-879, 886-891, 1160->1166, 1167-1199, 1202->1207, 1232-1243, 1272->1277, 1288->1290, 1351-1437, 1484->1488
security	Security-focused tests (required)	2026-02-26T19:09:16.6178765Z meow_decoder/decorrelation.py               47      0     20      6    91%   80->76, 127->131, 131->136, 136->140, 140->144, 144->exit
security	Security-focused tests (required)	2026-02-26T19:09:16.6178965Z meow_decoder/decoy_generator.py             73     73     10   

In [5]:
import subprocess

# Get all failures for each job
# First, get Security CI failure - search for lines with FAILED, ERROR, assertion
result = subprocess.run(
    ["gh", "run", "view", "22456719093", "--log-failed"],
    capture_output=True, text=True, cwd="/workspaces/meow-decoder"
)
log = result.stdout or result.stderr
lines = log.split('\n')
# Find lines containing FAILED or ERROR or assert or exit code
key_lines = [l for l in lines if any(x in l for x in ['FAILED', 'ERROR', 'error', 'exit', 'failed', 'RUSTSEC', 'CVE', 'Advisory', 'vulnerability'])]
print("\n".join(key_lines[:80]))


security	Security-focused tests (required)	2026-02-26T19:00:14.9884537Z   echo "Security tests failed!"
security	Security-focused tests (required)	2026-02-26T19:00:14.9884772Z   exit 1
security	Security-focused tests (required)	2026-02-26T19:00:27.9727147Z tests/security/test_dontdump.py::TestSecureMemoryDontdump::test_secure_memory_zeros_on_exit PASSED [  4%]
security	Security-focused tests (required)	2026-02-26T19:00:31.5074256Z tests/security/test_ratchet_forward_secrecy.py::TestForwardSecrecy::test_chain_keys_are_one_way FAILED [ 13%]
security	Security-focused tests (required)	2026-02-26T19:00:31.5324654Z tests/security/test_ratchet_forward_secrecy.py::TestForwardSecrecy::test_message_key_does_not_reveal_chain_key FAILED [ 14%]
security	Security-focused tests (required)	2026-02-26T19:00:31.5576772Z tests/security/test_ratchet_forward_secrecy.py::TestForwardSecrecy::test_subkeys_independent_per_frame FAILED [ 14%]
security	Security-focused tests (required)	2026-02-26T19:00:31.613791

In [6]:
import subprocess

# Get Security CI failure - only the most relevant lines
result = subprocess.run(
    ["gh", "run", "view", "22456719093", "--log-failed"],
    capture_output=True, text=True, cwd="/workspaces/meow-decoder"
)
log = result.stdout or result.stderr
lines = log.split('\n')
# Find lines containing 'FAILED' specifically
failed_lines = [l for l in lines if 'FAILED' in l or 'AssertionError' in l or 'Error:' in l or 'RUSTSEC' in l or 'vulnerability' in l.lower()]
print(f"Failed-related lines ({len(failed_lines)}):")
for l in failed_lines[:50]:
    print(l[-200:])  # last 200 chars only


Failed-related lines (150):
security	Security-focused tests (required)	2026-02-26T19:00:31.5074256Z tests/security/test_ratchet_forward_secrecy.py::TestForwardSecrecy::test_chain_keys_are_one_way FAILED [ 13%]
security	Security-focused tests (required)	2026-02-26T19:00:31.5324654Z tests/security/test_ratchet_forward_secrecy.py::TestForwardSecrecy::test_message_key_does_not_reveal_chain_key FAILED [ 14%]
security	Security-focused tests (required)	2026-02-26T19:00:31.5576772Z tests/security/test_ratchet_forward_secrecy.py::TestForwardSecrecy::test_subkeys_independent_per_frame FAILED [ 14%]
Security-focused tests (required)	2026-02-26T19:00:31.6137917Z tests/security/test_ratchet_forward_secrecy.py::TestPropertyBasedForwardSecrecy::test_brute_force_backward_derivation_fails FAILED [ 14%]
y-focused tests (required)	2026-02-26T19:00:31.6389169Z tests/security/test_ratchet_forward_secrecy.py::TestPropertyBasedForwardSecrecy::test_random_key_material_never_matches_early_keys FAILED [ 14%]
fo

In [7]:
# Use already-fetched log variable
lines = log.split('\n')
# Find specific failure patterns
print("=== FAILED tests ===")
for i, l in enumerate(lines):
    if 'FAILED tests/' in l:
        # Print surrounding context
        start = max(0, i-1)
        end = min(len(lines), i+2)
        for ll in lines[start:end]:
            print(ll[-300:])
        print("---")

print("\n=== short test summary ===")
in_summary = False
for l in lines:
    if 'short test summary' in l.lower():
        in_summary = True
    if in_summary:
        print(l[-200:])
        if l.strip().startswith('=') and in_summary and 'short' not in l.lower():
            break


=== FAILED tests ===
security	Security-focused tests (required)	2026-02-26T19:09:16.6193161Z =========================== short test summary info ============================
4097Z FAILED tests/security/test_ratchet_forward_secrecy.py::TestForwardSecrecy::test_chain_keys_are_one_way - RuntimeError: export_key() is PRODUCTION-FORBIDDEN. Use handle-based AEAD/HMAC operations instead. Set MEOW_PRODUCTION_MODE=0 to use in tests (MEOW_TEST_MODE alone is no longer sufficient).
sts/security/test_ratchet_forward_secrecy.py::TestForwardSecrecy::test_message_key_does_not_reveal_chain_key - RuntimeError: export_key() is PRODUCTION-FORBIDDEN. Use handle-based AEAD/HMAC operations instead. Set MEOW_PRODUCTION_MODE=0 to use in tests (MEOW_TEST_MODE alone is no longer sufficient).
---
4097Z FAILED tests/security/test_ratchet_forward_secrecy.py::TestForwardSecrecy::test_chain_keys_are_one_way - RuntimeError: export_key() is PRODUCTION-FORBIDDEN. Use handle-based AEAD/HMAC operations instead. Set MEOW_P

In [8]:
# Extract key failures from log
lines = log.split('\n')

# Find lines that have FAILED test names (short format)
# gh log format: "job_name\tstep_name\tTIMESTAMP CONTENT"
failed_test_names = []
cargo_audit_issues = []
pip_audit_issues = []
for l in lines:
    # Remove the prefix (job\tstep\ttimestamp )
    parts = l.split('\t', 2)
    content = parts[-1].strip() if len(parts) > 2 else l.strip()
    if content.startswith('FAILED tests/'):
        failed_test_names.append(content[:200])
    elif 'RUSTSEC-' in l or ('Crate:' in l and 'Severity:' not in lines[lines.index(l)-1] if l in lines else False):
        cargo_audit_issues.append(l[-200:])
    elif 'vulnerability' in l.lower() and ('ID:' in l or 'Advisory:' in l or 'Found' in l.lower()):
        pip_audit_issues.append(l[-200:])

print(f"Failed tests: {len(failed_test_names)}")
for t in failed_test_names[:30]:
    print(t)

print(f"\nCargo audit issues: {len(cargo_audit_issues)}")
for t in cargo_audit_issues[:10]:
    print(t)

print(f"\nPip audit issues: {len(pip_audit_issues)}")
for t in pip_audit_issues[:10]:
    print(t)


Failed tests: 0

Cargo audit issues: 0

Pip audit issues: 0


In [9]:
import subprocess

# Get all jobs for Security CI run
result = subprocess.run(
    ["gh", "run", "view", "22456719093"],
    capture_output=True, text=True, cwd="/workspaces/meow-decoder"
)
print(result.stdout)



X main Security CI · 22456719093
Triggered via push about 57 minutes ago

JOBS
✓ sbom in 1m35s (ID 65039771812)
X security in 10m53s (ID 65039771818)
  ✓ Set up job
  ✓ Checkout
  ✓ Set up Python 3.12
  ✓ Install system dependencies
  ✓ Install Python deps
  ✓ Run dtolnay/rust-toolchain@d0592fe69e35bc8f12e3dbaf9ad2694d976cb8e3
  ✓ Build + install Rust crypto backend (maturin)
  ✓ Verify Rust backend import
  ✓ Bandit security lint (Python static analysis)
  X Security-focused tests (required)
  - Post Set up Python 3.12
  ✓ Post Checkout
  ✓ Complete job
✓ dependency-audit in 7m13s (ID 65039771824)
✓ cargo-deny in 3m49s (ID 65039771870)
✓ pq-feature-gate in 29s (ID 65039771901)

ANNOTATIONS
X Process completed with exit code 1.
security: .github#4044


ARTIFACTS
sbom-artifacts

To see what failed, try: gh run view 22456719093 --log-failed
View this run on GitHub: https://github.com/systemslibrarian/meow-decoder/actions/runs/22456719093



In [10]:
import subprocess

# Get security job logs directly
result = subprocess.run(
    ["gh", "run", "view", "--job", "65039771818", "--log"],
    capture_output=True, text=True, cwd="/workspaces/meow-decoder"
)
log2 = result.stdout or result.stderr
lines2 = log2.split('\n')
print(f"Total lines: {len(lines2)}")
# Find the last 80 lines
last_lines = lines2[-80:]
for l in last_lines:
    # Strip the timestamp prefix
    parts = l.split('\t', 2)
    content = parts[-1].strip() if len(parts) > 2 else l.strip()
    print(content[:250])


Total lines: 5424
2026-02-26T19:09:16.6188762Z meow_decoder/source_cleanup.py              53      4     18      4    89%   93, 119, 175-176, 179->183
2026-02-26T19:09:16.6189172Z meow_decoder/stego_advanced.py             164    118     32      1    24%   57-67, 85-92, 140-150, 154, 179-208, 229-249, 264-289, 302-305, 309-325, 338-362, 376-391, 440, 455-472, 504-542
2026-02-26T19:09:16.6189491Z meow_decoder/stego_gif_binary.py           175    124     66      0    21%   64, 69, 74, 100, 104-106, 146-274, 298-312, 324-329, 348-375, 384-392, 401, 406
2026-02-26T19:09:16.6191315Z meow_decoder/stego_multilayer.py          1483   1288    538      1    10%   79-81, 90, 95, 251-268, 273-281, 290-317, 327-329, 334-336, 341-343, 369, 396-440, 456-518, 547-632, 637-642, 663-664, 682-793, 811-839, 857-858
2026-02-26T19:09:16.6191515Z meow_decoder/tamper_detection.py           199    199     70      0     0%   24-527
2026-02-26T19:09:16.6191706Z meow_decoder/tamper_report.py              107     

In [11]:
lines2 = log2.split('\n')
print(f"Total lines in security job log: {len(lines2)}")
# Just the last 60 non-empty lines
nonempty = [l for l in lines2 if l.strip()]
for l in nonempty[-60:]:
    parts = l.split('\t', 2)
    content = parts[-1].strip() if len(parts) > 2 else l.strip()
    if content:
        print(content[:300])


Total lines in security job log: 5424
2026-02-26T19:09:16.6200264Z FAILED tests/test_asymmetric_rekey.py::TestAsymmetricRekeyPrimitives::test_generate_recover_roundtrip - RuntimeError: export_key() is PRODUCTION-FORBIDDEN. Use handle-based AEAD/HMAC operations instead. Set MEOW_PRODUCTION_MODE=0 to use in tests (MEOW_TEST_MODE alone is 
2026-02-26T19:09:16.6201176Z FAILED tests/test_asymmetric_rekey.py::TestAsymmetricRekeyPrimitives::test_shared_secret_is_32_bytes - RuntimeError: export_key() is PRODUCTION-FORBIDDEN. Use handle-based AEAD/HMAC operations instead. Set MEOW_PRODUCTION_MODE=0 to use in tests (MEOW_TEST_MODE alone is n
2026-02-26T19:09:16.6202081Z FAILED tests/test_asymmetric_rekey.py::TestAsymmetricRekeyPrimitives::test_wrong_private_key_fails - RuntimeError: export_key() is PRODUCTION-FORBIDDEN. Use handle-based AEAD/HMAC operations instead. Set MEOW_PRODUCTION_MODE=0 to use in tests (MEOW_TEST_MODE alone is no 
2026-02-26T19:09:16.6202950Z FAILED tests/test_asymmetric_r

In [12]:
lines2 = log2.split('\n')
nonempty = [l for l in lines2 if l.strip()]
# Strip timestamps and write last 100 lines to a temp file
with open('/tmp/ci_last_lines.txt', 'w') as f:
    for l in nonempty[-100:]:
        parts = l.split('\t', 2)
        content = parts[-1].strip() if len(parts) > 2 else l.strip()
        if content:
            f.write(content[:300] + '\n')
print("Written to /tmp/ci_last_lines.txt")
print(f"Total non-empty lines: {len(nonempty)}")


Written to /tmp/ci_last_lines.txt
Total non-empty lines: 5423


In [68]:
import subprocess, os
os.chdir("/workspaces/meow-decoder")
r = subprocess.run(["git", "push", "origin", "main"], capture_output=True, text=True, timeout=60)
print("STDOUT:", r.stdout)
print("STDERR:", r.stderr)
print("RC:", r.returncode)

STDOUT: 
STDERR: To https://github.com/systemslibrarian/meow-decoder
   b24df10..20d2e0e  main -> main

RC: 0


In [ ]:
import subprocess, os
os.chdir("/workspaces/meow-decoder")
os.environ["MEOW_TEST_MODE"] = "1"

r = subprocess.run(
    ["python3", "-m", "pytest", "tests/", "-q", "--tb=line", "-x"],
    capture_output=True, text=True, timeout=300
)
lines = r.stdout.split('\n')
print('\n'.join(lines[-40:]))
if r.returncode != 0 and r.stderr:
    print("STDERR:", r.stderr[-500:])

Rust: test result: ok. 72 passed; 0 failed; 0 ignored; 0 measured; 0 filtered out; finished in 0.13s
test result: ok. 29 passed; 0 failed; 0 ignored; 0 measured; 0 filtered out; finished in 0.53s
test result: ok. 90 passed; 0 failed; 0 ignored; 0 measured; 0 filtered out; finished in 0.54s
test result: ok. 19 passed; 0 failed; 0 ignored; 0 measured; 0 filtered out; finished in 9.57s
test result: ok. 74 passed; 0 failed; 0 ignored; 0 measured; 0 filtered out; finished in 0.01s
test result: ok. 14 passed; 0 failed; 0 ignored; 0 measured; 0 filtered out; finished in 3.00s
test result: ok. 23 passed; 0 failed; 0 ignored; 0 measured; 0 filtered out; finished in 13.17s
test result: ok. 0 passed; 0 failed; 0 ignored; 0 measured; 0 filtered out; finished in 0.00s
------------------------------------------------------------------------------------
TOTAL                                     5588   4960   1656     75    11%
Coverage HTML written to dir htmlcov
============================= 126 pas

In [8]:
import subprocess, os
os.chdir("/workspaces/meow-decoder")
os.environ["MEOW_TEST_MODE"] = "1"

r = subprocess.run(
    ["python3", "-m", "pytest", "tests/test_stego_phase0.py", "-v", "--tb=short", "-x"],
    capture_output=True, text=True, timeout=120
)
# Print just the last part of the output
lines = r.stdout.split('\n')
# Find test results section
for i, line in enumerate(lines):
    if 'PASSED' in line or 'FAILED' in line or 'ERROR' in line or 'test_' in line or 'ERRORS' in line or '=====' in line:
        break
print('\n'.join(lines[max(0,i-2):]))
if r.returncode != 0:
    print("STDERR:", r.stderr[-3000:])

BUILD: OK
IMPORT: OK - all Phase 0 classes available
  CHANNEL_DISPOSAL=0x4, CHANNEL_COMMENT=0x5


In [2]:
import subprocess, os, sys

# Phase 1 Step 0: Verify environment and imports
env = dict(os.environ, MEOW_TEST_MODE="1", PYTHONPATH="/workspaces/meow-decoder")
r = subprocess.run(
    [sys.executable, "-c", """
import sys
print(f"Python: {sys.version}")

# Test all required imports
from meow_decoder.stego_multilayer import (
    MultiLayerStegoEncoder, MultiLayerStegoDecoder,
    MultiLayerConfig, validate_stego, CoercionLevel,
    ADVERSARIAL_STRENGTH_OFF, ADVERSARIAL_STRENGTH_LOW,
    ADVERSARIAL_STRENGTH_MEDIUM, ADVERSARIAL_STRENGTH_HIGH,
)
print("stego_multilayer imports OK")

import imageio.v3 as iio
print("imageio OK")

import numpy as np
from PIL import Image
print("numpy + PIL OK")

# Check carriers
import glob
carriers = sorted(glob.glob("/workspaces/meow-decoder/assets/cat*.jpg"))
print(f"Carriers found: {carriers}")

# Check Rust backend
try:
    import meow_crypto_rs
    print(f"meow_crypto_rs OK - has stego: {hasattr(meow_crypto_rs, 'stego_derive_frame_seed')}")
except ImportError:
    print("meow_crypto_rs NOT available (Python fallback)")
"""],
    env=env, capture_output=True, text=True, timeout=30
)
print(r.stdout)
if r.stderr:
    print("STDERR:", r.stderr[-500:])

Python: 3.11.14 (main, Jan 13 2026, 06:08:32) [GCC 12.2.0]
stego_multilayer imports OK
imageio OK
numpy + PIL OK
Carriers found: ['/workspaces/meow-decoder/assets/cat1.jpg', '/workspaces/meow-decoder/assets/cat2.jpg', '/workspaces/meow-decoder/assets/cat3.jpg', '/workspaces/meow-decoder/assets/cat4.jpg', '/workspaces/meow-decoder/assets/cat5.jpg']
meow_crypto_rs OK - has stego: True



In [1]:
print("kernel alive")

kernel alive


In [ ]:
import subprocess, os, sys
env = dict(os.environ, MEOW_TEST_MODE="1", PYTHONPATH="/workspaces/meow-decoder")
r = subprocess.run(
    [sys.executable, "/workspaces/meow-decoder/_audit_phase1_generate.py"],
    env=env, capture_output=True, text=True, timeout=600, cwd="/workspaces/meow-decoder"
)
print(r.stdout[-8000:] if len(r.stdout) > 8000 else r.stdout)
if r.returncode != 0:
    print(f"\nRETURN CODE: {r.returncode}")
    print("STDERR:", r.stderr[-3000:])

In [1]:
import subprocess, os, sys
env = dict(os.environ, MEOW_TEST_MODE="1", PYTHONPATH="/workspaces/meow-decoder")
# Quick single-artifact test
r = subprocess.run(
    [sys.executable, "-c", """
import os, hashlib, numpy as np, traceback
os.environ["MEOW_TEST_MODE"] = "1"
from pathlib import Path
from PIL import Image
import imageio.v3 as iio
from meow_decoder.stego_multilayer import (
    MultiLayerStegoEncoder, MultiLayerStegoDecoder,
    MultiLayerConfig, validate_stego,
)

OUT = Path("/workspaces/meow-decoder/_audit_artifacts")
OUT.mkdir(exist_ok=True)

# Create carrier GIF from cat1.jpg
img = Image.open("/workspaces/meow-decoder/assets/cat1.jpg").convert("RGB").resize((320, 240))
arr = np.array(img)
rng = np.random.RandomState(42)
frames = []
for i in range(10):
    noise = rng.normal(0, 0.5, arr.shape).astype(np.int16)
    frame = np.clip(arr.astype(np.int16) + noise, 0, 255).astype(np.uint8)
    frames.append(frame)
cover = OUT / "test_cover.gif"
iio.imwrite(str(cover), np.stack(frames), duration=100, loop=0)
print(f"Cover created: {cover.stat().st_size} bytes")

# Create stego
key = hashlib.sha256(b"test_key").digest()
payload = bytes(rng.randint(0, 256, size=4096, dtype=np.uint8))
config = MultiLayerConfig()
print(f"Config: channels={config.enable_primary},{config.enable_secondary},{config.enable_tertiary},{config.enable_disposal},{config.enable_comment},{config.enable_temporal}")
print(f"  immunize={config.immunize}, adversarial={config.adversarial_strength}, stc={config.use_stc}")

stego_path = OUT / "test_single.gif"
try:
    encoder = MultiLayerStegoEncoder(config, key)
    meta = encoder.encode(payload, str(cover), str(stego_path))
    print(f"Encode OK: {stego_path.stat().st_size} bytes")
    print(f"Meta: {meta}")
except Exception as e:
    traceback.print_exc()
    print(f"ENCODE FAILED: {e}")
    raise

# Validate
try:
    vr = validate_stego(str(stego_path), str(cover))
    print(f"Steganalysis: {vr.summary}")
except Exception as e:
    traceback.print_exc()
    print(f"VALIDATE FAILED: {e}")

# Decode
try:
    decoder = MultiLayerStegoDecoder(config, key)
    result = decoder.decode(str(stego_path))
    print(f"Decode: mac_valid={result.mac_valid}, channels={result.channel_sources}")
    print(f"Roundtrip match: {result.payload_bytes == payload}")
    print(f"Payload sizes - got: {len(result.payload_bytes)}, expected: {len(payload)}")
except Exception as e:
    traceback.print_exc()
    print(f"DECODE FAILED: {e}")
"""],
    env=env, capture_output=True, text=True, timeout=120, cwd="/workspaces/meow-decoder"
)
print(r.stdout)
if r.returncode != 0:
    print(f"RETURN CODE: {r.returncode}")
    print("STDERR:", r.stderr[-3000:])

TimeoutExpired: Command '['/workspaces/meow-decoder/.venv/bin/python', '-c', '\nimport os, hashlib, numpy as np, traceback\nos.environ["MEOW_TEST_MODE"] = "1"\nfrom pathlib import Path\nfrom PIL import Image\nimport imageio.v3 as iio\nfrom meow_decoder.stego_multilayer import (\n    MultiLayerStegoEncoder, MultiLayerStegoDecoder,\n    MultiLayerConfig, validate_stego,\n)\n\nOUT = Path("/workspaces/meow-decoder/_audit_artifacts")\nOUT.mkdir(exist_ok=True)\n\n# Create carrier GIF from cat1.jpg\nimg = Image.open("/workspaces/meow-decoder/assets/cat1.jpg").convert("RGB").resize((320, 240))\narr = np.array(img)\nrng = np.random.RandomState(42)\nframes = []\nfor i in range(10):\n    noise = rng.normal(0, 0.5, arr.shape).astype(np.int16)\n    frame = np.clip(arr.astype(np.int16) + noise, 0, 255).astype(np.uint8)\n    frames.append(frame)\ncover = OUT / "test_cover.gif"\niio.imwrite(str(cover), np.stack(frames), duration=100, loop=0)\nprint(f"Cover created: {cover.stat().st_size} bytes")\n\n# 

In [1]:
import subprocess, os, sys
env = dict(os.environ, MEOW_TEST_MODE="1", PYTHONPATH="/workspaces/meow-decoder")
r = subprocess.run(
    [sys.executable, "-c", """
import os, hashlib, numpy as np, time, traceback
os.environ["MEOW_TEST_MODE"] = "1"
from pathlib import Path
from PIL import Image
import imageio.v3 as iio
from meow_decoder.stego_multilayer import (
    MultiLayerStegoEncoder, MultiLayerConfig,
)

OUT = Path("/workspaces/meow-decoder/_audit_artifacts")
OUT.mkdir(exist_ok=True)

# Minimal carrier: 3 frames, 160x120
img = Image.open("/workspaces/meow-decoder/assets/cat1.jpg").convert("RGB").resize((160, 120))
arr = np.array(img)
frames = np.stack([arr]*3)
cover = OUT / "mini_cover.gif"
iio.imwrite(str(cover), frames, duration=100, loop=0)
print(f"Cover: {cover.stat().st_size} bytes, 3 frames 160x120")

key = hashlib.sha256(b"test").digest()
payload = b"x" * 64  # tiny

# Test 1: primary only, no STC, no immunize, no adversarial
t0 = time.time()
config = MultiLayerConfig(
    enable_primary=True, enable_secondary=False, enable_tertiary=False,
    enable_disposal=False, enable_comment=False, enable_temporal=False,
    use_stc=False, immunize=False, adversarial_strength=0,
)
encoder = MultiLayerStegoEncoder(config, key)
stego = OUT / "mini_test1.gif"
meta = encoder.encode(payload, str(cover), str(stego))
print(f"Test1 (primary, no STC/immune/adv): {time.time()-t0:.2f}s, meta={meta}")

# Test 2: primary only WITH STC
t0 = time.time()
config2 = MultiLayerConfig(
    enable_primary=True, enable_secondary=False, enable_tertiary=False,
    enable_disposal=False, enable_comment=False, enable_temporal=False,
    use_stc=True, immunize=False, adversarial_strength=0,
)
encoder2 = MultiLayerStegoEncoder(config2, key)
stego2 = OUT / "mini_test2.gif"
meta2 = encoder2.encode(payload, str(cover), str(stego2))
print(f"Test2 (primary+STC): {time.time()-t0:.2f}s, meta={meta2}")

# Test 3: primary + immunize
t0 = time.time()
config3 = MultiLayerConfig(
    enable_primary=True, enable_secondary=False, enable_tertiary=False,
    enable_disposal=False, enable_comment=False, enable_temporal=False,
    use_stc=True, immunize=True, adversarial_strength=0,
)
encoder3 = MultiLayerStegoEncoder(config3, key)
stego3 = OUT / "mini_test3.gif"
meta3 = encoder3.encode(payload, str(cover), str(stego3))
print(f"Test3 (primary+STC+immunize): {time.time()-t0:.2f}s, meta={meta3}")

# Test 4: all channels, full config
t0 = time.time()
config4 = MultiLayerConfig()
encoder4 = MultiLayerStegoEncoder(config4, key)
stego4 = OUT / "mini_test4.gif"
meta4 = encoder4.encode(payload, str(cover), str(stego4))
print(f"Test4 (full 6-chan config): {time.time()-t0:.2f}s, meta={meta4}")

print("DONE")
"""],
    env=env, capture_output=True, text=True, timeout=300, cwd="/workspaces/meow-decoder"
)
print(r.stdout)
if r.returncode != 0:
    print(f"RC: {r.returncode}")
    print("STDERR:", r.stderr[-3000:])

Cover: 18868 bytes, 3 frames 160x120
Test1 (primary, no STC/immune/adv): 0.17s, meta={'channels_used': ['primary'], 'payload_size': 64, 'prepared_size': 86, 'primary_bits': 688, 'secondary_bits': 0, 'tertiary_bits': 0, 'disposal_bits': 0, 'comment_bytes': 0, 'psnr': 70.23274491236029, 'total_capacity': 688, 'immunized': False, 'saliency_costs': True, 'temporal_bits': 0, 'adversarial_strength': 0}
Test2 (primary+STC): 7.69s, meta={'channels_used': ['primary'], 'payload_size': 64, 'prepared_size': 86, 'primary_bits': 688, 'secondary_bits': 0, 'tertiary_bits': 0, 'disposal_bits': 0, 'comment_bytes': 0, 'psnr': 70.05301120224128, 'total_capacity': 688, 'immunized': False, 'saliency_costs': True, 'temporal_bits': 0, 'adversarial_strength': 0}
Test3 (primary+STC+immunize): 7.70s, meta={'channels_used': ['primary'], 'payload_size': 64, 'prepared_size': 86, 'primary_bits': 688, 'secondary_bits': 0, 'tertiary_bits': 0, 'disposal_bits': 0, 'comment_bytes': 0, 'psnr': 70.53674850515404, 'total_ca

In [2]:
import subprocess, os, sys
env = dict(os.environ, MEOW_TEST_MODE="1", PYTHONPATH="/workspaces/meow-decoder")

# Run fast artifacts first: baseline + all non-STC (ids 0, 6-22, 24)
fast_ids = [0] + list(range(6, 23)) + [24]
for aid in fast_ids:
    r = subprocess.run(
        [sys.executable, "/workspaces/meow-decoder/_audit_phase1_batch.py", str(aid)],
        env=env, capture_output=True, text=True, timeout=120, cwd="/workspaces/meow-decoder"
    )
    # Print last 500 chars of output
    out = r.stdout.strip()
    if out:
        lines = out.split('\n')
        for line in lines[:8]:  # First 8 lines per artifact
            print(line)
    if r.returncode != 0:
        print(f"  FAILED (rc={r.returncode}): {r.stderr[-300:]}")
    print()
print("Fast batch complete!")

[0/25] Generating: baseline_plain
  Plain carrier, no stego (control)
  Baseline: FAIL: RS: PASS (p=0.012) | Chi^2: PASS (det=0.000, p=0.0000) | SPA: DETECTED (rate=0.973)

Total artifacts defined: 25

[6/25] Generating: ml_primary_cat1
  Primary LSB only, no STC, 2KB, cat1
  Encoded in 1.7s: 135655 bytes
  PSNR=35.5dB SSIM=0.9973
  Steg: FAIL: RS: PASS (p=0.030) | Chi^2: PASS (det=0.000, p=0.0000) | SPA: DETECTED (rate=0.984)
  Decoded in 1.0s: RT=False MAC=False CH=['primary']
  STATUS: FAIL(RT)


[7/25] Generating: ml_primary_cat3
  Primary LSB only, no STC, 3KB, cat3
  Encoded in 1.5s: 119374 bytes
  PSNR=37.2dB SSIM=0.9992
  Steg: FAIL: RS: PASS (p=0.012) | Chi^2: PASS (det=0.000, p=0.0000) | SPA: DETECTED (rate=0.977)
  Decoded in 0.9s: RT=False MAC=False CH=['primary']
  STATUS: FAIL(RT)


[8/25] Generating: ml_decoy_cat1
  Decoy key, shallow, 1KB, cat1
  Encoded in 1.6s: 135203 bytes
  PSNR=35.5dB SSIM=0.9973
  Steg: FAIL: RS: PASS (p=0.023) | Chi^2: PASS (det=0.000, p=0.0000) 

In [ ]:
import subprocess, os, sys
env = dict(os.environ, MEOW_TEST_MODE="1", PYTHONPATH="/workspaces/meow-decoder")
# Quick single-artifact test with APNG
r = subprocess.run(
    [sys.executable, "-c", """
import os, hashlib, numpy as np, time, traceback
os.environ["MEOW_TEST_MODE"] = "1"
from pathlib import Path
from PIL import Image
import imageio.v3 as iio
from meow_decoder.stego_multilayer import (
    MultiLayerStegoEncoder, MultiLayerStegoDecoder,
    MultiLayerConfig, validate_stego,
)

OUT = Path("/workspaces/meow-decoder/_audit_artifacts")
OUT.mkdir(exist_ok=True)

# Create APNG carrier from cat1.jpg
img = Image.open("/workspaces/meow-decoder/assets/cat1.jpg").convert("RGB").resize((200, 150))
arr = np.array(img)
rng = np.random.RandomState(42)
frames = []
for i in range(5):
    noise = rng.normal(0, 0.5, arr.shape).astype(np.int16)
    frame = np.clip(arr.astype(np.int16) + noise, 0, 255).astype(np.uint8)
    frames.append(frame)

cover = OUT / "test_cover.png"
pil_frames = [Image.fromarray(f, "RGB") for f in frames]
pil_frames[0].save(str(cover), format="PNG", save_all=True,
                   append_images=pil_frames[1:], duration=100, loop=0)
print(f"APNG cover: {cover.stat().st_size} bytes")

# Verify APNG preserves pixels
readback = iio.imread(str(cover), index=None)
if isinstance(readback, np.ndarray) and readback.ndim == 4:
    diff = np.abs(readback[0].astype(int) - frames[0].astype(int))
    print(f"APNG pixel preservation: max_diff={diff.max()}, mean_diff={diff.mean():.4f}")

# Encode stego
key = hashlib.sha256(b"test_key").digest()
payload = bytes(rng.randint(0, 256, size=2048, dtype=np.uint8))

# Primary-only, no STC for speed
config = MultiLayerConfig(
    enable_secondary=False, enable_tertiary=False,
    enable_disposal=False, enable_comment=False,
    enable_temporal=False, use_stc=False, immunize=False,
    adversarial_strength=0,
)

stego = OUT / "test_apng_stego.png"
t0 = time.time()
encoder = MultiLayerStegoEncoder(config, key)
meta = encoder.encode(payload, str(cover), str(stego))
print(f"Encode: {time.time()-t0:.2f}s, meta={meta}")
print(f"Stego: {stego.stat().st_size} bytes")

# Verify stego pixels preserved in APNG
stego_read = iio.imread(str(stego), index=None)
if isinstance(stego_read, np.ndarray) and stego_read.ndim == 4:
    print(f"Stego APNG frames: {stego_read.shape}")

# Decode
decoder = MultiLayerStegoDecoder(config, key)
result = decoder.decode(str(stego))
print(f"Decode: mac={result.mac_valid}, channels={result.channel_sources}")
print(f"Roundtrip: {result.payload_bytes == payload}")
if not result.payload_bytes == payload:
    print(f"  payload len: got={len(result.payload_bytes)} expected={len(payload)}")
    # Check first bytes
    got = result.payload_bytes[:20]
    exp = payload[:20]
    print(f"  got[:20] = {got.hex()}")
    print(f"  exp[:20] = {exp.hex()}")

# Steganalysis
vr = validate_stego(str(stego), str(cover))
print(f"Steganalysis: {vr.summary}")
"""],
    env=env, capture_output=True, text=True, timeout=60, cwd="/workspaces/meow-decoder"
)
print(r.stdout)
if r.returncode != 0:
    print(f"RC: {r.returncode}")
    print("STDERR:", r.stderr[-2000:])

In [13]:
import subprocess

# Get the most recent fuzz workflow run details
result = subprocess.run(
    ["gh", "run", "list", "--workflow=fuzz.yml", "--limit=5", "--json", "databaseId,status,conclusion,displayTitle,createdAt"],
    capture_output=True, text=True
)
print(result.stdout[:3000])
print(result.stderr[:500] if result.stderr else "")


[
  {
    "conclusion": "failure",
    "createdAt": "2026-02-26T18:58:24Z",
    "databaseId": 22456719137,
    "displayTitle": "docs: remove previous release APK link from README",
    "status": "completed"
  },
  {
    "conclusion": "failure",
    "createdAt": "2026-02-26T18:57:26Z",
    "databaseId": 22456683251,
    "displayTitle": "docs: add Android APK download links + iOS/store coming-soon notices …",
    "status": "completed"
  },
  {
    "conclusion": "failure",
    "createdAt": "2026-02-26T18:33:19Z",
    "databaseId": 22455822118,
    "displayTitle": "chore: track Android APKs with Git LFS",
    "status": "completed"
  },
  {
    "conclusion": "failure",
    "createdAt": "2026-02-26T18:31:13Z",
    "databaseId": 22455745309,
    "displayTitle": "fix(android): add missing splash_logo drawable, rebuild v3.2.1 APK",
    "status": "completed"
  },
  {
    "conclusion": "failure",
    "createdAt": "2026-02-26T18:27:44Z",
    "databaseId": 22455619494,
    "displayTitle": "fix(andr

In [14]:
import subprocess

# Get fuzz job IDs
result = subprocess.run(
    ["gh", "run", "view", "22456719137", "--json", "jobs"],
    capture_output=True, text=True
)
import json
data = json.loads(result.stdout)
for job in data['jobs']:
    print(f"Job: {job['name']} | ID: {job['databaseId']} | Status: {job['conclusion']}")
    for step in job.get('steps', []):
        if step.get('conclusion') in ('failure', 'timed_out', None):
            print(f"  FAILED step: {step['name']} | conclusion: {step.get('conclusion')}")


JSONDecodeError: Expecting value: line 1 column 1 (char 0)

In [15]:
import subprocess, json

# Get fuzz job log (last 150 non-empty lines)
result = subprocess.run(
    ["gh", "run", "view", "--log-failed", "22456719137"],
    capture_output=True, text=True
)
lines = [l for l in result.stdout.splitlines() if l.strip()]
# Show last 120 lines
for l in lines[-120:]:
    print(l)
print("---STDERR---")
print(result.stderr[:500] if result.stderr else "(none)")


Python Fuzzing (Atheris)	Fuzz multi-layer stego	2026-02-26T19:32:21.7358044Z INFO: Instrumenting numpy._core._string_helpers
Python Fuzzing (Atheris)	Fuzz multi-layer stego	2026-02-26T19:32:21.7381535Z INFO: Instrumenting numpy._core._type_aliases
Python Fuzzing (Atheris)	Fuzz multi-layer stego	2026-02-26T19:32:21.7494167Z INFO: Instrumenting numpy._core.einsumfunc
Python Fuzzing (Atheris)	Fuzz multi-layer stego	2026-02-26T19:32:21.8766213Z INFO: Instrumenting numpy._core.numeric
Python Fuzzing (Atheris)	Fuzz multi-layer stego	2026-02-26T19:32:21.9857787Z INFO: Instrumenting numpy._core.shape_base
Python Fuzzing (Atheris)	Fuzz multi-layer stego	2026-02-26T19:32:22.0279774Z INFO: Instrumenting numpy._core.fromnumeric
Python Fuzzing (Atheris)	Fuzz multi-layer stego	2026-02-26T19:32:22.1006314Z INFO: Instrumenting numpy._core._methods
Python Fuzzing (Atheris)	Fuzz multi-layer stego	2026-02-26T19:32:22.1403358Z INFO: Instrumenting pickle
Python Fuzzing (Atheris)	Fuzz multi-layer stego	2026

In [16]:
import subprocess

# Save fuzz logs to file, then read the last 200 non-empty lines
result = subprocess.run(
    ["gh", "run", "view", "--log-failed", "22456719137"],
    capture_output=True, text=True
)
with open("/tmp/fuzz_log.txt", "w") as f:
    f.write(result.stdout)

lines = [l for l in result.stdout.splitlines() if l.strip()]
print(f"Total non-empty lines: {len(lines)}")
# Show last 100
print("\n=== LAST 100 LINES ===")
for l in lines[-100:]:
    print(l)


Total non-empty lines: 164

=== LAST 100 LINES ===
Python Fuzzing (Atheris)	Fuzz multi-layer stego	2026-02-26T19:32:22.8879618Z INFO: Instrumenting ctypes
Python Fuzzing (Atheris)	Fuzz multi-layer stego	2026-02-26T19:32:22.9726798Z INFO: Instrumenting ctypes._endian
Python Fuzzing (Atheris)	Fuzz multi-layer stego	2026-02-26T19:32:22.9813681Z INFO: Instrumenting numpy._pytesttester
Python Fuzzing (Atheris)	Fuzz multi-layer stego	2026-02-26T19:32:22.9932203Z INFO: Instrumenting numpy.lib
Python Fuzzing (Atheris)	Fuzz multi-layer stego	2026-02-26T19:32:22.9981471Z INFO: Instrumenting numpy.lib._arraypad_impl
Python Fuzzing (Atheris)	Fuzz multi-layer stego	2026-02-26T19:32:23.0612160Z INFO: Instrumenting numpy.lib._index_tricks_impl
Python Fuzzing (Atheris)	Fuzz multi-layer stego	2026-02-26T19:32:23.1460483Z INFO: Instrumenting numpy.matrixlib
Python Fuzzing (Atheris)	Fuzz multi-layer stego	2026-02-26T19:32:23.1469400Z INFO: Instrumenting numpy.matrixlib.defmatrix
Python Fuzzing (Atheris)	

In [17]:
import subprocess

# Get just the error summary from fuzz logs
result = subprocess.run(
    ["grep", "-E", "(Error|FAILED|error:|RuntimeError|exit code|Traceback|assert|fuzz_|atheris)", "/tmp/fuzz_log.txt"],
    capture_output=True, text=True
)
lines = result.stdout.splitlines()
# Show last 80 error lines
print(f"Total error-like lines: {len(lines)}")
for l in lines[-80:]:
    # Strip the github job prefix (format: "jobname\tstepname\t content")
    parts = l.split('\t', 2)
    print(parts[-1] if len(parts) >= 3 else l)


Total error-like lines: 8
2026-02-26T19:32:20.9109698Z timeout ${DURATION}s python fuzz/fuzz_stego_multilayer.py || RC=$?
2026-02-26T19:32:26.2019211Z TypeError: MultiLayerConfig.__init__() got an unexpected keyword argument 'lsb_depth'
2026-02-26T19:32:26.2022645Z Traceback (most recent call last):
2026-02-26T19:32:26.2031503Z   File "/home/runner/work/meow-decoder/meow-decoder/fuzz/fuzz_stego_multilayer.py", line 278, in combined_fuzz
2026-02-26T19:32:26.2032233Z     fuzz_validate_stego(data)
2026-02-26T19:32:26.2032939Z   File "/home/runner/work/meow-decoder/meow-decoder/fuzz/fuzz_stego_multilayer.py", line 227, in fuzz_validate_stego
2026-02-26T19:32:26.2034520Z TypeError: MultiLayerConfig.__init__() got an unexpected keyword argument 'lsb_depth'
2026-02-26T19:32:26.2322148Z ##[error]Process completed with exit code 77.


In [18]:
import subprocess, json

# Get the most recent run of each workflow
result = subprocess.run(
    ["gh", "run", "list", "--limit=30", "--json", "workflowName,conclusion,databaseId,createdAt,status"],
    capture_output=True, text=True
)
runs = json.loads(result.stdout)

# Group by workflow name, show most recent
seen = {}
for r in runs:
    name = r['workflowName']
    if name not in seen:
        seen[name] = r

print(f"{'Workflow':<50} {'Status':<12} {'Conclusion':<12}")
print("-" * 76)
for name, r in sorted(seen.items()):
    print(f"{name:<50} {r['status']:<12} {r.get('conclusion','?'):<12}")


JSONDecodeError: Expecting value: line 1 column 1 (char 0)

In [19]:
import subprocess

# Use text output from gh run list
result = subprocess.run(
    ["gh", "run", "list", "--limit=40"],
    capture_output=True, text=True
)
print(result.stdout[:4000])
print(result.stderr[:300] if result.stderr else "")


completed	failure	docs: remove previous release APK link from README	Fuzzing	main	push	22456719137	34m5s	2026-02-26T18:58:24Z
completed	failure	docs: remove previous release APK link from README	CI - Tests + Coverage	main	push	22456719130	1h0m25s	2026-02-26T18:58:24Z
completed	success	docs: remove previous release APK link from README	CodeQL	main	push	22456719126	10m13s	2026-02-26T18:58:24Z
completed	failure	docs: remove previous release APK link from README	Security CI	main	push	22456719093	10m58s	2026-02-26T18:58:24Z
completed	success	docs: remove previous release APK link from README	OpenSSF Scorecard	main	push	22456719087	47s	2026-02-26T18:58:24Z
completed	success	docs: remove previous release APK link from README	Deploy to GitHub Pages	main	push	22456719068	3m13s	2026-02-26T18:58:24Z
completed	failure	docs: remove previous release APK link from README	.github/workflows/rust-test-coverage.yml	main	push	22456718082	0s	2026-02-26T18:58:22Z
completed	failure	docs: add Android APK down

In [20]:
import subprocess

# Check if uv is available or can be installed
result = subprocess.run(["which", "uv"], capture_output=True, text=True)
print("uv location:", result.stdout.strip() or "NOT FOUND")

# Check if pip-tools/pip-compile is available
result2 = subprocess.run(["which", "pip-compile"], capture_output=True, text=True)
print("pip-compile:", result2.stdout.strip() or "NOT FOUND")

# Try to install uv
result3 = subprocess.run(
    ["pip", "install", "uv", "--quiet"],
    capture_output=True, text=True
)
print("uv install:", result3.returncode, result3.stdout[-200:] if result3.stdout else "", result3.stderr[-200:] if result3.stderr else "")

result4 = subprocess.run(["which", "uv"], capture_output=True, text=True)
print("uv location after install:", result4.stdout.strip() or "NOT FOUND")


uv location: NOT FOUND
pip-compile: NOT FOUND
uv install: 0   A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: pip install --upgrade pip

uv location after install: /workspaces/meow-decoder/.venv/bin/uv


In [21]:
import subprocess

# Regenerate requirements-ci.lock
print("Regenerating requirements-ci.lock ...")
result = subprocess.run(
    ["/workspaces/meow-decoder/.venv/bin/uv", "pip", "compile",
     "--generate-hashes",
     "requirements-ci.in",
     "-o", "requirements-ci.lock"],
    capture_output=True, text=True,
    cwd="/workspaces/meow-decoder"
)
print("Exit code:", result.returncode)
if result.returncode == 0:
    print("SUCCESS - last 20 lines:")
    lines = result.stdout.splitlines() if result.stdout else []
    for l in lines[-20:]:
        print(l)
else:
    print("FAILED:")
    print(result.stderr[-2000:] if result.stderr else "")
    print(result.stdout[-1000:] if result.stdout else "")


Regenerating requirements-ci.lock ...
Exit code: 0
SUCCESS - last 20 lines:
tzdata==2025.3 \
    --hash=sha256:06a47e5700f3081aab02b2e513160914ff0694bce9947d6b76ebd6bf57cfc5d1 \
    --hash=sha256:de39c2ca5dc7b0344f2eba86f49d614019d29f060fc4ebc8a417896a620b56a7
    # via arrow
uri-template==1.3.0 \
    --hash=sha256:0e00f8eb65e18c7de20d595a14336e9f337ead580c70934141624b6d1ffdacc7 \
    --hash=sha256:a44a133ea12d44a0c0f06d7d42a52d71282e77e2f937d8abd5655b8d56fc1363
    # via jsonschema
urllib3==2.6.3 \
    --hash=sha256:1b62b6884944a57dbe321509ab94fd4d3b307075e0c2eae991ac71ee15ad38ed \
    --hash=sha256:bf272323e553dfb2e87d9bfd225ca7b0f467b919d7bbd355436d3fd37cb0acd4
    # via requests
wcwidth==0.6.0 \
    --hash=sha256:1a3a1e510b553315f8e146c54764f4fb6264ffad731b3d78088cdb1478ffbdad \
    --hash=sha256:cdc4e4262d6ef9a1a57e018384cbeb1208d8abbc64176027e2c2455c81313159
    # via prettytable
webcolors==25.10.0 \
    --hash=sha256:032c727334856fc0b968f63daa252a1ac93d33db2f5267756623c210e57a4f

In [22]:
import subprocess

uv = "/workspaces/meow-decoder/.venv/bin/uv"
cwd = "/workspaces/meow-decoder"

for in_file, out_file in [
    ("requirements.txt", "requirements.lock"),
    ("requirements-dev.txt", "requirements-dev.lock"),
]:
    print(f"Regenerating {out_file} ...")
    result = subprocess.run(
        [uv, "pip", "compile", "--generate-hashes", in_file, "-o", out_file],
        capture_output=True, text=True, cwd=cwd
    )
    if result.returncode == 0:
        print(f"  ✅ {out_file} regenerated")
    else:
        print(f"  ❌ FAILED (exit {result.returncode}):")
        print(result.stderr[-1000:] if result.stderr else "")
        print(result.stdout[-500:] if result.stdout else "")


Regenerating requirements.lock ...
  ✅ requirements.lock regenerated
Regenerating requirements-dev.lock ...
  ✅ requirements-dev.lock regenerated


In [23]:
import subprocess

for lock_file in ["requirements.lock", "requirements-dev.lock", "requirements-ci.lock"]:
    result = subprocess.run(
        ["grep", "-E", "^cryptography==|^pillow==|^maturin==",
         f"/workspaces/meow-decoder/{lock_file}"],
        capture_output=True, text=True
    )
    versions = result.stdout.strip()
    print(f"{lock_file}: {versions if versions else '(not in this file)'}")


requirements.lock: cryptography==46.0.5 \
pillow==12.1.1 \
requirements-dev.lock: cryptography==46.0.4 \
requirements-ci.lock: cryptography==46.0.5 \
maturin==1.12.4 \


In [24]:
import subprocess

cwd = "/workspaces/meow-decoder"

files = [
    ".github/workflows/ci.yml",
    ".github/workflows/security-ci.yml",
    ".github/workflows/fuzz.yml",
    ".github/workflows/long-fuzz.yml",
    ".github/workflows/pyinstaller.yml",
    ".github/workflows/rust-security-suite.yml",
    ".github/workflows/rust-crypto.yml",
    ".github/workflows/mutation-testing.yml",
    ".github/workflows/rust-test-coverage.yml",
    "fuzz/fuzz_stego_multilayer.py",
    "requirements-ci.lock",
    "requirements.lock",
    "requirements-dev.lock",
]

r = subprocess.run(["git", "add"] + files, capture_output=True, text=True, cwd=cwd)
print("git add:", r.returncode, r.stderr if r.stderr else "OK")

r = subprocess.run(
    ["git", "commit", "-m",
     "fix(ci): add MEOW_PRODUCTION_MODE=0 to all test workflows, fix rust-test-coverage.yml, fix fuzz lsb_bits, regenerate lock files\n\n"
     "- Add MEOW_PRODUCTION_MODE=0 alongside MEOW_TEST_MODE=1 in all 8 CI\n"
     "  workflows that run Python tests; export_key() now requires both\n"
     "  (fixes 50 test failures in Security CI and main CI)\n"
     "- Fix rust-test-coverage.yml: Generate coverage badge step was missing\n"
     "  its `- name:` header causing a duplicate `run:` key YAML error (0s failure)\n"
     "- Fix fuzz/fuzz_stego_multilayer.py: MultiLayerConfig field renamed\n"
     "  lsb_depth -> lsb_bits, use_timing_channel -> enable_secondary,\n"
     "  use_palette_channel -> enable_tertiary (fixes Fuzzing workflow TypeError)\n"
     "- Regenerate requirements-ci.lock (cryptography 45->46.0.5, maturin 1.11->1.12.4),\n"
     "  requirements.lock (cryptography 46.0.4->46.0.5, pillow 12.1.0->12.1.1),\n"
     "  requirements-dev.lock"],
    capture_output=True, text=True, cwd=cwd
)
print("git commit:", r.returncode)
print(r.stdout)
if r.stderr:
    print(r.stderr)

r = subprocess.run(["git", "push"], capture_output=True, text=True, cwd=cwd)
print("git push:", r.returncode, r.stdout, r.stderr if r.stderr else "")


git add: 0 OK
git commit: 0
[main 5297687] fix(ci): add MEOW_PRODUCTION_MODE=0 to all test workflows, fix rust-test-coverage.yml, fix fuzz lsb_bits, regenerate lock files
 Author: Paul Clark <systemslibrarian@gmail.com>
 13 files changed, 363 insertions(+), 676 deletions(-)

git push: 0  To https://github.com/systemslibrarian/meow-decoder
   9808196..5297687  main -> main

